## spark envoirment setup

In [ ]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null

In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"

In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local[*]").appName("ColabPySpark").getOrCreate()

In [ ]:
spark.version

'3.5.1'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MergeWeatherFlights") \
    .config("spark.executor.memory", "2g") \
    .getOrCreate()

## weather dataset get combined

In [ ]:
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, to_date, year

# Start Spark session
spark = SparkSession.builder.appName("WeatherFlightJoinClean").getOrCreate()

# Load 2023 Excel weather data
weather_files = [
    "Boston Weather Data.xlsx",
    "Chicago Weather Data.xlsx",
    "New York Weather Data.xlsx",
    "San Francisco Weather Data.xlsx",
    "Washington Weather Data.xlsx"
]

weather_dfs = []

for file in weather_files:
    path = f"/content/drive/MyDrive/Project/{file}"
    df_pd = pd.read_excel(path)
    df_pd.dropna(axis=1, how='all', inplace=True)
    df_pd.dropna(axis=0, how='all', inplace=True)

    # Rename column if needed
    if "datetime" in df_pd.columns:
        df_pd.rename(columns={"datetime": "StartTime(UTC)"}, inplace=True)

    # Add city info
    df_pd['City'] = file.split()[0]
    df_pd = df_pd.replace({np.nan: None})

    df_spark = spark.createDataFrame(df_pd)
    weather_dfs.append(df_spark)

# Union all 2023 weather DataFrames
df_weather_2023 = weather_dfs[0]
for df in weather_dfs[1:]:
    df_weather_2023 = df_weather_2023.unionByName(df, allowMissingColumns=True)

# Load 2016–2022 weather CSV
df_weather_2016_2022 = spark.read.csv(
    "/content/drive/MyDrive/Project/WeatherEvents_Jan2016-Dec2022.csv",
    header=True,
    inferSchema=True
)

# Filter 2016–2022 data to only years 2019–2022
df_weather_2016_2022 = df_weather_2016_2022.withColumn("EventYear", year(to_date("StartTime(UTC)")))
df_weather_2016_2022 = df_weather_2016_2022.filter((col("EventYear") >= 2019) & (col("EventYear") <= 2022)).drop("EventYear")

# Add missing columns to 2023 weather
cols_to_add = set(df_weather_2016_2022.columns) - set(df_weather_2023.columns)
for col_name in cols_to_add:
    df_weather_2023 = df_weather_2023.withColumn(col_name, lit(None))

# Reorder 2023 columns to match 2016–2022
df_weather_2023 = df_weather_2023.select(df_weather_2016_2022.columns)

# Combine filtered 2016–2022 weather and 2023 weather
df_all_weather = df_weather_2016_2022.unionByName(df_weather_2023)

# Convert StartTime(UTC) to EventDate
df_all_weather = df_all_weather.withColumn("EventDate", to_date("StartTime(UTC)"))

# Select only necessary weather columns
df_all_weather_clean = df_all_weather.select(
    "EventDate", "City", "Type", "Severity", "Precipitation(in)"
)

# Load flight data
df_flights = spark.read.csv(
    "/content/drive/MyDrive/Project/flights_sample_3m.csv",
    header=True,
    inferSchema=True
)

# Add FlightDate column and select relevant flight columns
df_flights = df_flights.withColumn("FlightDate", to_date("FL_DATE"))

# Join flights with cleaned weather data
df_final = df_flights.join(
    df_all_weather_clean,
    (df_flights["ORIGIN_CITY"] == df_all_weather_clean["City"]) &
    (df_flights["FlightDate"] == df_all_weather_clean["EventDate"]),
    how="left"
)

# Show a few rows with selected columns
df_final.select(
    "FL_DATE", "ORIGIN_CITY", "FlightDate", "EventDate", "Type", "Severity", "Precipitation(in)"
).show(10, truncate=False)


+----------+-------------------+----------+---------+----+--------+-----------------+
|FL_DATE   |ORIGIN_CITY        |FlightDate|EventDate|Type|Severity|Precipitation(in)|
+----------+-------------------+----------+---------+----+--------+-----------------+
|2022-01-22|Anchorage, AK      |2022-01-22|NULL     |NULL|NULL    |NULL             |
|2021-06-11|Atlanta, GA        |2021-06-11|NULL     |NULL|NULL    |NULL             |
|2021-04-12|Baltimore, MD      |2021-04-12|NULL     |NULL|NULL    |NULL             |
|2020-12-28|Fort Lauderdale, FL|2020-12-28|NULL     |NULL|NULL    |NULL             |
|2021-08-05|Fort Lauderdale, FL|2021-08-05|NULL     |NULL|NULL    |NULL             |
|2021-01-21|Greer, SC          |2021-01-21|NULL     |NULL|NULL    |NULL             |
|2020-03-09|Helena, MT         |2020-03-09|NULL     |NULL|NULL    |NULL             |
|2023-04-11|Houston, TX        |2023-04-11|NULL     |NULL|NULL    |NULL             |
|2019-07-08|Huntsville, AL     |2019-07-08|NULL     |N

In [ ]:
df_final.schema

StructType([StructField('FL_DATE', DateType(), True), StructField('AIRLINE', StringType(), True), StructField('AIRLINE_DOT', StringType(), True), StructField('AIRLINE_CODE', StringType(), True), StructField('DOT_CODE', IntegerType(), True), StructField('FL_NUMBER', IntegerType(), True), StructField('ORIGIN', StringType(), True), StructField('ORIGIN_CITY', StringType(), True), StructField('DEST', StringType(), True), StructField('DEST_CITY', StringType(), True), StructField('CRS_DEP_TIME', IntegerType(), True), StructField('DEP_TIME', DoubleType(), True), StructField('DEP_DELAY', DoubleType(), True), StructField('TAXI_OUT', DoubleType(), True), StructField('WHEELS_OFF', DoubleType(), True), StructField('WHEELS_ON', DoubleType(), True), StructField('TAXI_IN', DoubleType(), True), StructField('CRS_ARR_TIME', IntegerType(), True), StructField('ARR_TIME', DoubleType(), True), StructField('ARR_DELAY', DoubleType(), True), StructField('CANCELLED', DoubleType(), True), StructField('CANCELLATIO

In [ ]:
# Save the final DataFrame as a single CSV file
output_path = "/content/drive/MyDrive/Project/final_joined_weather_flight.csv"

df_final.coalesce(1).write.csv(
    path=output_path,
    mode="overwrite",
    header=True
)
